<a href="https://colab.research.google.com/github/braga-sadewa/ISYS2001-A2/blob/main/22347705_ISYS2001_A2_CODE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# JOB 1 - USER PROFILE
# ============================================================

user_profile = {
    "name": "",
    "currency_label": ""
}


def set_user_name(name):
    # Store the user's name
    user_profile["name"] = name.strip()

    return f"Hello, {user_profile['name']}!"


def set_currency_label(currency_label):
    # Store the user's currency label
    user_profile["currency_label"] = currency_label.strip()

    return f"Currency label set to {user_profile['currency_label']}."


def change_name(new_name):
    # Change the existing name
    user_profile["name"] = new_name.strip()

    return f"Name changed to {user_profile['name']}."


def change_currency_label(new_currency_label):
    # Change the existing currency label
    user_profile["currency_label"] = new_currency_label.strip()

    return f"Currency label changed to {user_profile['currency_label']}."


def user_profile_setup(name, currency_label):
    # Set the user's initial profile
    set_user_name(name)
    set_currency_label(currency_label)

    return (
        f"Hello, {user_profile['name']}! "
        f"Welcome to your personalized finance tracker. "
        f"Your currency label is {user_profile['currency_label']}. "
        f"Let's get started!"
    )

In [ ]:
# ============================================================
# JOB 2 - ADD A TRANSACTION
# ============================================================

from datetime import datetime

# Store all transactions
transactions = []

# Keep track of the next transaction ID
next_transaction_id = 1


def create_transaction(date, category, account, amount, description):
    global next_transaction_id

    # Validate the date
    try:
        datetime.strptime(date, "%Y-%m-%d")
    except ValueError:
        return "Invalid date. Please use YYYY-MM-DD."

    # Validate the amount
    try:
        amount = float(amount)
    except (ValueError, TypeError):
        return "Invalid amount. Please enter a number."

    # Zero is neither income nor expense
    if amount == 0:
        return "Invalid amount. Please enter a non-zero number."

    # Generate a unique transaction ID
    transaction_id = "T" + str(next_transaction_id)
    next_transaction_id += 1

    # Create the transaction record
    transaction = {
        "id": transaction_id,
        "date": date,
        "category": category,
        "account": account,
        "amount": amount,
        "description": description
    }

    # Store the transaction
    transactions.append(transaction)

    # Confirm that the transaction was recorded
    return f"Transaction {transaction_id} recorded successfully."


def add_transaction(date, category, account, amount, description):
    return create_transaction(
        date,
        category,
        account,
        amount,
        description
    )


def recall_state():
    # If there are no transactions yet, there is nothing to recall
    if len(transactions) == 0:
        return {
            "last_date": None,
            "last_category": None,
            "last_account": None,
            "categories": [],
            "accounts": []
        }

    # Get the most recently added transaction
    last_transaction = transactions[-1]

    # Build lists of previously used categories and accounts
    categories = []
    accounts = []

    for transaction in transactions:

        if transaction["category"] not in categories:
            categories.append(transaction["category"])

        if transaction["account"] not in accounts:
            accounts.append(transaction["account"])

    return {
        "last_date": last_transaction["date"],
        "last_category": last_transaction["category"],
        "last_account": last_transaction["account"],
        "categories": categories,
        "accounts": accounts
    }

In [ ]:
# ============================================================
# JOB 3 - VIEW AND DELETE TRANSACTIONS
# ============================================================


def view_transaction():

    transaction_list = []

    # Organise each transaction into a list
    for transaction in transactions:

        row = [
            transaction["id"],
            transaction["date"],
            transaction["category"],
            transaction["account"],
            transaction["amount"],
            transaction["description"]
        ]

        transaction_list.append(row)

    return transaction_list


def select_transaction(transaction_id):

    for transaction in transactions:

        if transaction["id"] == transaction_id:
            return transaction

    return None


def delete_transaction(transaction_id):

    transaction = select_transaction(transaction_id)

    if transaction is None:
        return "Transaction not found."

    transactions.remove(transaction)

    return f"Transaction {transaction_id} deleted."

In [ ]:
# ============================================================
# JOB 4 - ANALYSE TRANSACTIONS
# ============================================================

import pandas as pd


def analyse_transactions():

    # Convert stored transaction dictionaries into a DataFrame
    df = pd.DataFrame(transactions)

    # Handle the case where there are no transactions
    if df.empty:
        return {
            "total_income": 0,
            "total_expense": 0,
            "net_cash_flow": 0,
            "monthly": pd.DataFrame(),
            "annual": pd.DataFrame(),
            "expense_by_category": pd.Series(dtype="float64"),
            "expense_category_percentage": pd.Series(dtype="float64"),
            "account_analysis": pd.DataFrame()
        }

    # Convert the date column into a proper pandas date
    df["date"] = pd.to_datetime(df["date"])

    # Create separate income and expense columns
    df["income"] = 0.0
    df["expense"] = 0.0

    # Positive amounts are income
    df.loc[df["amount"] > 0, "income"] = df.loc[
        df["amount"] > 0,
        "amount"
    ]

    # Negative amounts are expenses, displayed as positive values
    df.loc[df["amount"] < 0, "expense"] = df.loc[
        df["amount"] < 0,
        "amount"
    ].abs()

    # Overall calculations
    total_income = df["income"].sum()
    total_expense = df["expense"].sum()

    net_cash_flow = total_income - total_expense

    # Monthly calculations
    df["month"] = df["date"].dt.to_period("M").astype(str)

    monthly = df.groupby("month")[["income", "expense"]].sum()

    monthly["net_cash_flow"] = (
        monthly["income"] - monthly["expense"]
    )

    # Annual calculations
    df["year"] = df["date"].dt.year

    annual = df.groupby("year")[["income", "expense"]].sum()

    annual["net_cash_flow"] = (
        annual["income"] - annual["expense"]
    )

    # Expense by category
    expense_by_category = (
        df[df["amount"] < 0]
        .groupby("category")["expense"]
        .sum()
    )

    # Expense percentage by category
    if total_expense > 0:
        expense_category_percentage = (
            expense_by_category / total_expense
        ) * 100
    else:
        expense_category_percentage = pd.Series(
            dtype="float64"
        )

    # Income and expense by account
    account_analysis = (
        df.groupby("account")[["income", "expense"]]
        .sum()
    )

    account_analysis["net_cash_flow"] = (
        account_analysis["income"]
        - account_analysis["expense"]
    )

    # Return all analysis results
    return {
        "total_income": total_income,
        "total_expense": total_expense,
        "net_cash_flow": net_cash_flow,
        "monthly": monthly,
        "annual": annual,
        "expense_by_category": expense_by_category,
        "expense_category_percentage": expense_category_percentage,
        "account_analysis": account_analysis
    }

In [ ]:
# ============================================================
# JOB 5 - GENERATE DASHBOARD
# ============================================================

import matplotlib.pyplot as plt


def generate_dashboard(user_question=None):

    # Get the calculated results from Job 4
    analysis = analyse_transactions()

    # Get the user's currency label
    currency = user_profile["currency_label"]

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    summary = (
        f"Total Income: {currency}{analysis['total_income']:.2f}\n"
        f"Total Expense: {currency}{analysis['total_expense']:.2f}\n"
        f"Net Cash Flow: {currency}{analysis['net_cash_flow']:.2f}"
    )

    # --------------------------------------------------------
    # Monthly income vs expense
    # --------------------------------------------------------

    monthly = analysis["monthly"]

    if monthly.empty:

        monthly_figure = None

    else:

        monthly_figure, ax = plt.subplots(
            figsize=(10, 5)
        )

        monthly[["income", "expense"]].plot(
            kind="bar",
            ax=ax
        )

        ax.set_title("Monthly Income vs Expense")
        ax.set_xlabel("Month")
        ax.set_ylabel(f"Amount ({currency})")
        ax.tick_params(axis="x", rotation=45)
        ax.legend(["Income", "Expense"])

        monthly_figure.tight_layout()

    # --------------------------------------------------------
    # Annual income vs expense
    # --------------------------------------------------------

    annual = analysis["annual"]

    if annual.empty:

        annual_figure = None

    else:

        annual_figure, ax = plt.subplots(
            figsize=(8, 5)
        )

        annual[["income", "expense"]].plot(
            kind="bar",
            ax=ax
        )

        ax.set_title("Annual Income vs Expense")
        ax.set_xlabel("Year")
        ax.set_ylabel(f"Amount ({currency})")
        ax.tick_params(axis="x", rotation=0)
        ax.legend(["Income", "Expense"])

        annual_figure.tight_layout()

    # --------------------------------------------------------
    # Expense by category
    # --------------------------------------------------------

    expense_by_category = analysis["expense_by_category"]

    expense_category_percentage = (
        analysis["expense_category_percentage"]
    )

    if expense_by_category.empty:

        category_figure = None
        category_text = "No expense data available."

    else:

        category_figure, ax = plt.subplots(
            figsize=(7, 7)
        )

        ax.pie(
            expense_by_category,
            labels=expense_by_category.index,
            autopct="%1.1f%%",
            startangle=90
        )

        # Create a hole in the middle
        centre_circle = plt.Circle(
            (0, 0),
            0.70,
            fc="white"
        )

        ax.add_artist(centre_circle)

        ax.set_title("Expense by Category")

        category_figure.tight_layout()

        # Create the small category summary
        category_lines = []

        for category in expense_by_category.index:

            amount = expense_by_category[category]
            percentage = expense_category_percentage[category]

            category_lines.append(
                f"{category}: "
                f"{currency}{amount:.2f} "
                f"({percentage:.1f}%)"
            )

        category_text = "\n".join(category_lines)

    # --------------------------------------------------------
    # Income / Expense by Account
    # --------------------------------------------------------

    account_analysis = (
        analysis["account_analysis"]
        .round(2)
        .reset_index()
    )

    if account_analysis.empty:

        account_table = pd.DataFrame(
            columns=[
                "Account",
                "Income",
                "Expense",
                "Net Cash Flow"
            ]
        )

    else:

        account_analysis.columns = [
            "Account",
            "Income",
            "Expense",
            "Net Cash Flow"
        ]

        account_table = account_analysis

    # --------------------------------------------------------
    # Optional AI response
    # --------------------------------------------------------

    ai_response = ""

    if user_question:

        ai_response = ask_gemini(
            user_question,
            analysis
        )

    # --------------------------------------------------------
    # Return dashboard components
    # --------------------------------------------------------

    return {
        "summary": summary,
        "monthly_figure": monthly_figure,
        "annual_figure": annual_figure,
        "category_figure": category_figure,
        "category_text": category_text,
        "account_table": account_table,
        "ai_response": ai_response
    }

In [ ]:
# ============================================================
# JOB 6 - AI ASSISTANT
# ============================================================

# Run these in Google Colab if needed:
!pip install -q -U google-genai

from google import genai
from google.colab import userdata

api_key = userdata.get("GOOGLE_API_KEY")

client = genai.Client(
    api_key=api_key
)

MODEL = "gemini-3.5-flash"


finance_assistant_personality = """
You are a friendly and approachable finance assistant built into a personal
finance tracker.

Your purpose is to help users understand and interpret their own financial data,
especially summaries, trends, categories, and account information provided by
the application.

When the user's question relates to their financial data, use the provided data
to give a personalised response. Do not invent financial figures that are not
provided.

If the user asks a general finance-related question that cannot be answered
from their personal data, you may provide general educational information,
but clearly state that it is generic information rather than professional
financial advice.

If the user asks something unrelated to finance, politely explain that you are
designed to help with financial questions and redirect the conversation toward
the finance tracker.

Use a casual, friendly, and approachable tone rather than a rigid professional
financial-advisor tone.

Keep explanations beginner-friendly and concise while still providing enough
information to be useful.

Begin naturally with a friendly greeting when appropriate, such as "Hi!",
and end naturally with a friendly phrase such as "Take care!" when appropriate.

Do not present your responses as professional financial advice.
"""


def ask_gemini(user_question, analysis):

    # Convert the analysis results into text
    financial_context = f"""
Financial summary:
Total income: {analysis["total_income"]}
Total expense: {analysis["total_expense"]}
Net cash flow: {analysis["net_cash_flow"]}

Monthly analysis:
{analysis["monthly"].to_string()}

Annual analysis:
{analysis["annual"].to_string()}

Expense by category:
{analysis["expense_by_category"].to_string()}

Expense category percentages:
{analysis["expense_category_percentage"].to_string()}

Account analysis:
{analysis["account_analysis"].to_string()}
"""

    # Combine the financial context with the question
    prompt = f"""
Here is the user's current financial data:

{financial_context}

The user's question is:

{user_question}

Answer the user's question using the financial data when relevant.
Do not invent financial figures.
"""

    try:

        response = client.models.generate_content(
            model=MODEL,
            contents=prompt,
            config=genai.types.GenerateContentConfig(
                system_instruction=finance_assistant_personality
            )
        )

        return response.text

    except Exception as error:

        return f"Gemini API error: {error}"

In [ ]:
# ============================================================
# GRADIO - USER INTERFACE AND INTEGRATION
# ============================================================

# Run this in Google Colab if needed:
# !pip install -q gradio

import gradio as gr


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def transaction_table():
    """Return transactions in the format needed by Gradio."""

    return view_transaction()


def transaction_ids():
    """Return the IDs currently available for deletion."""

    return [
        transaction["id"]
        for transaction in transactions
    ]


def refresh_transaction_controls():

    state = recall_state()

    return (
        gr.update(
            choices=state["categories"],
            value=state["last_category"]
        ),

        gr.update(
            choices=state["accounts"],
            value=state["last_account"]
        ),

        gr.update(
            choices=transaction_ids(),
            value=None
        )
    )


def refresh_dashboard_outputs():

    dashboard = generate_dashboard()

    return (
        dashboard["summary"],
        dashboard["monthly_figure"],
        dashboard["annual_figure"],
        dashboard["category_figure"],
        dashboard["category_text"],
        dashboard["account_table"]
    )


# ------------------------------------------------------------
# Gradio Add Transaction function
# ------------------------------------------------------------

def add_transaction_gradio(
    date,
    category,
    account,
    amount,
    description
):

    # Run the existing Job 2
    message = add_transaction(
        date,
        category,
        account,
        amount,
        description
    )

    # Refresh everything affected by a transaction
    return (
        message,

        # View Transaction
        transaction_table(),

        # Category dropdown,
        # Account dropdown,
        # Delete transaction dropdown
        *refresh_transaction_controls(),

        # View Report
        *refresh_dashboard_outputs()
    )


# ------------------------------------------------------------
# Gradio Delete Transaction function
# ------------------------------------------------------------

def delete_transaction_gradio(transaction_id):

    # Run the existing Job 3
    message = delete_transaction(transaction_id)

    return (
        message,

        # Refresh transaction table
        transaction_table(),

        # Refresh delete dropdown
        gr.update(
            choices=transaction_ids(),
            value=None
        ),

        # Refresh View Report
        *refresh_dashboard_outputs()
    )


# ------------------------------------------------------------
# Gradio Gemini function
# ------------------------------------------------------------

def ask_gemini_gradio(user_question):

    analysis = analyse_transactions()

    return ask_gemini(
        user_question,
        analysis
    )


# ============================================================
# BUILD THE GRADIO APP
# ============================================================

with gr.Blocks() as app:

    gr.Markdown("# Smart Finance Tracker")

    # --------------------------------------------------------
    # USER PROFILE TAB
    # --------------------------------------------------------

    with gr.Tab("User Profile"):

        gr.Markdown("## User Profile")

        name_input = gr.Textbox(
            label="Name",
            placeholder="Enter your name"
        )

        currency_input = gr.Textbox(
            label="Currency Label",
            placeholder="e.g. AUD, A$, $, IDR, Rp"
        )

        setup_button = gr.Button("Save Profile")

        profile_message = gr.Textbox(
            label="Profile Status",
            interactive=False
        )

        gr.Markdown("### Change Name")

        new_name_input = gr.Textbox(
            label="New Name"
        )

        change_name_button = gr.Button(
            "Change Name"
        )

        name_message = gr.Textbox(
            label="Name Status",
            interactive=False
        )

        gr.Markdown("### Change Currency Label")

        new_currency_input = gr.Textbox(
            label="New Currency Label"
        )

        change_currency_button = gr.Button(
            "Change Currency Label"
        )

        currency_message = gr.Textbox(
            label="Currency Status",
            interactive=False
        )

    # --------------------------------------------------------
    # ADD TRANSACTION TAB
    # --------------------------------------------------------

    with gr.Tab("Add Transaction"):

        gr.Markdown("## Add Transaction")

        date_input = gr.DateTime(
            label="Date",
            include_time=False,
            type="string"
        )

        category_input = gr.Dropdown(
            label="Category",
            choices=[],
            allow_custom_value=True
        )

        account_input = gr.Dropdown(
            label="Account",
            choices=[],
            allow_custom_value=True
        )

        amount_input = gr.Number(
            label="Amount",
            value=0
        )

        description_input = gr.Textbox(
            label="Description",
            placeholder="Describe the transaction"
        )

        add_button = gr.Button(
            "Add Transaction"
        )

        add_message = gr.Textbox(
            label="Confirmation",
            interactive=False
        )

    # --------------------------------------------------------
    # VIEW TRANSACTION TAB
    # --------------------------------------------------------

    with gr.Tab("View Transaction"):

        gr.Markdown("## View Transactions")

        transaction_table_display = gr.Dataframe(
            headers=[
                "ID",
                "Date",
                "Category",
                "Account",
                "Amount",
                "Description"
            ],
            value=transaction_table(),
            interactive=False
        )

        delete_id_input = gr.Dropdown(
            label="Select Transaction ID",
            choices=transaction_ids()
        )

        delete_button = gr.Button(
            "Delete Transaction"
        )

        delete_message = gr.Textbox(
            label="Confirmation",
            interactive=False
        )

    # --------------------------------------------------------
    # VIEW REPORT TAB
    # --------------------------------------------------------

    with gr.Tab("View Report"):

        gr.Markdown("## Financial Report")

        initial_dashboard = generate_dashboard()

        report_summary = gr.Textbox(
            label="Summary",
            value=initial_dashboard["summary"],
            lines=4,
            interactive=False
        )

        gr.Markdown(
            "### Monthly Income vs Expense"
        )

        monthly_chart = gr.Plot(
            value=initial_dashboard["monthly_figure"]
        )

        gr.Markdown(
            "### Annual Income vs Expense"
        )

        annual_chart = gr.Plot(
            value=initial_dashboard["annual_figure"]
        )

        gr.Markdown(
            "### Expense by Category"
        )

        category_chart = gr.Plot(
            value=initial_dashboard["category_figure"]
        )

        category_summary = gr.Textbox(
            label="Expense by Category",
            value=initial_dashboard["category_text"],
            interactive=False
        )

        gr.Markdown(
            "### Income / Expense by Account"
        )

        account_table = gr.Dataframe(
            headers=[
                "Account",
                "Income",
                "Expense",
                "Net Cash Flow"
            ],
            value=initial_dashboard["account_table"],
            interactive=False
        )

        gr.Markdown(
            "### Ask Finance Assistant"
        )

        ai_question = gr.Textbox(
            label="Your Question",
            placeholder="e.g. Which account has the highest expenses?"
        )

        ai_button = gr.Button(
            "Ask Finance Assistant"
        )

        ai_response = gr.Textbox(
            label="Finance Assistant",
            lines=8,
            interactive=False
        )

    # ========================================================
    # EVENT CONNECTIONS
    # ========================================================

    # User Profile
    setup_button.click(
        fn=user_profile_setup,
        inputs=[
            name_input,
            currency_input
        ],
        outputs=profile_message
    )

    change_name_button.click(
        fn=change_name,
        inputs=new_name_input,
        outputs=name_message
    )

    change_currency_button.click(
        fn=change_currency_label,
        inputs=new_currency_input,
        outputs=currency_message
    )

    # Add Transaction
    add_button.click(
        fn=add_transaction_gradio,
        inputs=[
            date_input,
            category_input,
            account_input,
            amount_input,
            description_input
        ],
        outputs=[
            # Confirmation
            add_message,

            # View Transaction
            transaction_table_display,

            # Recall state controls
            category_input,
            account_input,
            delete_id_input,

            # View Report
            report_summary,
            monthly_chart,
            annual_chart,
            category_chart,
            category_summary,
            account_table
        ]
    )

    # Delete Transaction
    delete_button.click(
        fn=delete_transaction_gradio,
        inputs=delete_id_input,
        outputs=[
            # Confirmation
            delete_message,

            # View Transaction
            transaction_table_display,
            delete_id_input,

            # View Report
            report_summary,
            monthly_chart,
            annual_chart,
            category_chart,
            category_summary,
            account_table
        ]
    )

    # AI Assistant
    ai_button.click(
        fn=ask_gemini_gradio,
        inputs=ai_question,
        outputs=ai_response
    )


# ============================================================
# LAUNCH
# ============================================================

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://153c4fd200f5a5fdd2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
